In [ ]:
import glob
import pandas as pd
import numpy as np
import scipy.stats
import matplotlib.pyplot as plt
import seaborn as sns

import h5py

import os.path as op
import os
from os.path import join, exists
import glob
import shutil
from datetime import datetime
import math

# from nilearn import plotting
# from nilearn import image
# from nilearn import masking
# import nilearn

import nibabel as nib

import scipy

import csv

from tqdm import tqdm

In [2]:
datestring = datetime.now()
print(datestring)
timestampStr = datestring.strftime("%b%d_%Y")
print(timestampStr)

2026-03-30 17:02:58.892165
Mar30_2026


The goal here should be to obtain betas for each emotion word for each voxel in the language localizer binary mask. I'm not entirely sure what export format would be best for this data type -- I don't know if it would be too small to fit a csv or if we should keep .nii or if we should use .npy or something.

The next step in data analysis is to compute an RSA matrix and then to be able to compare it to the behavioral matrix.

In [3]:
##define variables

sub_uid='sub-002' #CHANGE PER SUB

task_name= 'emotion_word' 

glmsingle_model = 'TYPED_FITHRF_GLMDENOISE_RR.hdf5'
exp_num_words = 24
exp_num_trials = 72
# exp_num_trs = 196

localizer_id = 'langlocSN'
# rois = {1:'LH_IFGorb', 2:'LH_IFG', 3:'LH_MFG', 4:'LH_AntTemp', 5:'LH_PostTemp', 6:'LH_AnG', 7:'RH_IFGorb', 8:'RH_IFG', 9:'RH_MFG', 10:'RH_AntTemp', 11:'RH_PostTemp', 12:'RH_AnG', 13: 'ALL'}
rois = {1: 'ALL'}

In [7]:
##set paths 
# TO CHANGE: change all paths to correct paths (ask where each thing is) - DONE

# set path to stimsets and stimset name
# path_to_stimsets = '/usr/people/bs1799/neu502b/neu502b_fmri/emotion_word_glmsingle/emotion_word_stim'
path_to_stimsets = '/Users/bianca/Desktop/NEU502B/neu502b_fmri/emotion_word_glmsingle/emotion_word_stim'
stimset_name = f'stimset_emotion_word_{sub_uid}.csv'
# path_to_stimsets = '/nese/mit/group/evlab/u/holson/EXPT_LBLLM/analyses/glmsingle/stimset_outputs/lbllm'
# stimset_name = f'stimset_lbllm_{sub_uid}_all_sessions.csv'

#path to design matrices
path_to_design_matrices = '/usr/people/bs1799/neu502b/neu502b_fmri/emotion_word_glmsingle/design_matrices'
# '/nese/mit/group/evlab/u/holson/EXPT_LBLLM/analyses/glmsingle/design_matrices/lbllm'

#path to fROI masks
path_to_fROI_mask = f'/usr/people/zt4569/neu502b/neu502b_fmri/localizer_masks/{sub_uid}'
# path_to_fROI_mask= f'/nese/mit/group/evlab/u/holson/LBLLM_analysis/output_localizers/langloc/top10percent/{sub_uid}_*_PL2017'

#path to glmsingle output
if sub_uid == "sub-004":
    path_to_glmsingle_output = f'/usr/people/bs1799/neu502b/neu502b_fmri/emotion_word_glmsingle/output_glmsingle/output_glmsingle_UID-{sub_uid}_pcstop-5_fracs-0.05_brainR2-0.7727'
else:
    path_to_glmsingle_output = f'/usr/people/bs1799/neu502b/neu502b_fmri/emotion_word_glmsingle/output_glmsingle/output_glmsingle_UID-{sub_uid}_pcstop-5_fracs-0.05'
# path_to_glmsingle_output= f'/nese/mit/group/evlab/u/holson/LBLLM_analysis/output_glmsingle/output_glmsingle/output_glmsingle_preproc-swr_pcstop5_fracs-0.05_UID-{sub_uid}'

#path to outputs
path_to_outputs = f'/usr/people/bs1799/neu502b/neu502b_fmri/emotion_word_glmsingle/output_betas'
# path_to_outputs= f'/nese/mit/group/evlab/u/holson/LBLLM_analysis/output_glmsingle/outputs_{timestampStr}/{sub_uid}'

In [12]:
## get order of itemIDs from stimset 
stimset_table = pd.read_csv(join(path_to_stimsets, stimset_name))

all_word_ids_in_order = list(map(str, list(map(int, list(stimset_table["word_id"])))))

# # open the design matrix - list of np arrays, one for each run 
# design_matrix = pd.read_pickle(join(path_to_design_matrices, design_matrix_name))

# # dictionary where each run is a key (by run_idx) and each value is an ordered list of the conditions in the run (by item_id)
# item_id_run_dict = {}
# # list of all item_ids in chronological order (should match glm_single output)
# all_item_ids_in_order = []

# for i, run_design_matrix in enumerate(design_matrix):
#     run_idx = i + 1 
    
#     item_id_run_dict[str(run_idx)] = []

#     # get shape of design matrix and check that it matches what we expect  
#     num_trs, num_conds = run_design_matrix.shape
#     assert num_trs == exp_num_trs, "unexpected number of time points in design matrix"
#     assert num_conds == exp_num_trials, "unexpected number of conditions in design matrix"

#     # for each tr in the design matrix, record the item id of the corresponding condition if there is a 1
#     for i, design_row in enumerate(run_design_matrix):
#         if np.sum(design_row) != 0:
#             item_id = str(np.argwhere(design_row != 0)[0, 0] + 1)
#             item_id_run_dict[str(run_idx)].append(item_id)
#             all_item_ids_in_order.append(item_id)

print(all_word_ids_in_order)

['1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '10', '8', '13', '16', '22', '17', '11', '15', '20', '4', '19', '23', '5', '6', '24', '21', '3', '1', '14', '12', '7', '9', '18', '2', '8', '24', '17', '14', '19', '10', '20', '6', '5', '18', '16', '15', '2', '9', '12', '3', '13', '1', '22', '21', '7', '23', '4', '11']


In [ ]:
## get word responses in language parcel

#load betas across all trials
glmsingle_model_file=h5py.File(op.join(path_to_glmsingle_output,glmsingle_model),'r')
emotion_word_betas_dataset=glmsingle_model_file['betasmd']

#check dimensions 
print(emotion_word_betas_dataset)

(x_dim, y_dim, z_dim, num_trials) = emotion_word_betas_dataset.shape

assert num_trials == exp_num_trials 

#convert to numpy array
emotion_word_trial_betas=np.zeros((x_dim, y_dim, z_dim, num_trials))
emotion_word_betas_dataset.read_direct(emotion_word_trial_betas)

# #create dictionary with raw betas for each fROI and item_id
raw_beta_means = {}


# TO DO:
# keep track of which presentation of the stimulus it is (1, 2, or 3) - done! 
# modify storage dictionary to keep track of stim presentation - done! 
# modify export file name to keep track of stim presentation - done! 
# add a last step that calculates average betas across all three stim presentations

#for each trial...
for trial_idx, item_id in enumerate(all_word_ids_in_order):
    
    if trial_idx < 24:
        repeat_idx = 1
    elif trial_idx >= 24 and trial_idx < 48:
        repeat_idx = 2
    else:
        repeat_idx = 3

    raw_beta_means[item_id] = {}
    
    # load trial beta map
    trial_beta_array_img = emotion_word_trial_betas[:,:,:,trial_idx]
       
    #for each fROI...    
    for roi_num, roi_label in rois.items():      
               
        # load fROI mask
        froi_mask_name = f'{roi_label}_fROI_mask_binary_top10percent.npy'
        froi_mask_img = np.load(glob.glob(join(path_to_fROI_mask,froi_mask_name))[0])
        
        # initialize output
        #output_array_item = np.zeros((x_dim, y_dim, z_dim))
        output_array_item = np.full((x_dim, y_dim, z_dim), np.nan)
        
        # find where mask is 1
        indices = np.where(froi_mask_img == 1)
        
        # get the corresponding values in trial beta map
        corresponding_values = trial_beta_array_img[indices]
        
        # assign the corresponding values to the output array at the specified indices
        output_array_item[indices] = corresponding_values
        
        # saving individual trial arrays        
        export_path = f'{path_to_outputs}/raw_betas/{roi_label}'
        export_filename = f'{export_path}/{task_name}_itemID_{item_id}_repeat_{repeat_idx}_{roi_label}_raw_betas'  
        # if no dir make dir
        if not exists(export_path):
            os.makedirs(export_path)
        # saving np array to .npy file
        np.save(f'{export_filename}.npy', output_array_item)
        
        #save non-normalized average betas per fROI        
        raw_beta_means[item_id][roi_label][repeat_idx] = output_array_item
        
        print('raw beta mean example', raw_beta_means[item_id][roi_label][repeat_idx])
        


In [ ]:
# #get session IDs for each item

# #consider updating code to make sure that item order is not dependent on spreadsheet order

# # initialize dictionary to map session ids to a map of the corresponding item ids in the order they occurred in 
# session_items_dict = {}
# session_items_dict["1"] = []
# session_items_dict["2"] = []

# # get the item_id and corresponding session_id from the stimset 
# with open(join(path_to_stimsets, stimset_name), mode = 'r', encoding='utf-8-sig') as file:
#     stimset_table = csv.DictReader(file, dialect = 'excel')     # may need to change dialect as needed

#     for row in stimset_table:
#         item_id = str(int(float(row['item_id'])))
#         session_id = row['session_id']

#         session_items_dict[session_id].append(item_id)

# # get length of each session list and print it
# num_items_session1 = len(session_items_dict["1"])
# num_items_session2 = len(session_items_dict["2"])
# print(f"There are {num_items_session1} items in session 1 and {num_items_session2} items in session 2")

# print(session_items_dict["1"])
# print(session_items_dict["2"])

# # get overall list of items in order and assert that it's the same as the ordered list from the design matrix
# all_item_ids_in_order_stimset = session_items_dict["1"] + session_items_dict["2"]
# assert all_item_ids_in_order_stimset == all_item_ids_in_order, "the order of item ids from the design matrix does not match the order of item ids from the stimset"


In [ ]:
# #z score betas by session

# #initialize output csv
# output_fname = f'{task_name}_responsesByItem_sub-{sub_uid}_{timestampStr}.csv'
# df_betas_perItem = op.join(path_to_outputs, output_fname)


# #initialize
# z_scored_array_mean_list = []


# for session_num in list(session_items_dict.keys()):
#     for roi_num in rois:
#         roi = rois[roi_num]
#         print(roi)
#         items_in_session = session_items_dict[session_num]
#         #items_in_session_str = [str(item) for item in items_in_session]
        
#         print('items in session',items_in_session)
        
#         #path to raw beta maps
#         path_to_raw_beta_maps = f'{path_to_outputs}/raw_betas/{roi}'
        
#         # list to store files matching each session
#         filtered_files = []
#         for itemid in items_in_session:
#             filtered_files.append(f'{task_name}_itemID_{itemid}_{roi}_raw_betas.npy')
#             print('item id', itemid)
            
#         #print(filtered_files)
        
#         # List to store the loaded numpy arrays
#         unstacked_raw_arrays = []

#         # Load each filtered .npy file and store the array
#         for file in tqdm(filtered_files, desc="Loading files"):
#             file_path = join(path_to_raw_beta_maps, file)
#             array = np.load(file_path)
#             unstacked_raw_arrays.append(array)
            
#         # stack them along a new axis to create a 4D array
#         stacked_raw_array = np.stack(unstacked_raw_arrays, axis=0)
        
        
#         #print('stacked array elements',np.unique(stacked_raw_array.reshape(-1)))
    

#         print(stacked_raw_array.shape)  # Output will depend on the number and shape of the loaded arrays
        
#         # Identify non-NaN voxels across trials (axis=0)
#         not_all_nan_voxels = ~np.isnan(stacked_raw_array).all(axis=0)

#         # Filter out voxels that are all NaN
#         filtered_array = stacked_raw_array[:, not_all_nan_voxels]

#         # Check for voxels with zero variance across trials
#         zero_variance_voxels = np.std(filtered_array, axis=0) == 0
        
#         zero_variance_count = np.sum(zero_variance_voxels)
#         print(f"Number of voxels with zero variance: {zero_variance_count}")


#         # Apply z-score normalization to the filtered array (excluding zero variance voxels)
#         z_score_filtered_array = scipy.stats.zscore(filtered_array[:, ~zero_variance_voxels], axis=0)

#         # Create an array of NaNs with the same shape as the original array
#         z_score_array = np.full(stacked_raw_array.shape, np.nan)

#         # Initialize a mask for non-NaN, non-zero variance voxels
#         final_mask = np.full(stacked_raw_array.shape[1:], False)
#         final_mask[not_all_nan_voxels] = ~zero_variance_voxels

#         # Fill the z-score array with the computed z-scores for non-NaN, non-zero variance voxels
#         z_score_array[:, final_mask] = z_score_filtered_array

#         # For zero variance voxels, set z-scores to 0
#         # Adjusted to handle the boolean indexing correctly
#         zero_variance_final_mask = np.full(stacked_raw_array.shape[1:], False)
#         zero_variance_final_mask[not_all_nan_voxels] = zero_variance_voxels
#         z_score_array[:, zero_variance_final_mask] = 0

        
        
#         # unstack it
#         unstacked_z_scored_array = np.split(z_score_array, z_score_array.shape[0], axis=0)
#         assert len(unstacked_z_scored_array) == len(items_in_session)

        
#         for idx, itemid in enumerate(items_in_session):
            
#             z_scored_array = unstacked_z_scored_array[idx]
#             #print('z scored array elements',np.unique(z_scored_array.reshape(-1)))
            
#             export_path = f'{path_to_outputs}/zscored_betas/{roi}'
#             export_filename = f'{export_path}/{task_name}_itemID_{itemid}_fROI_zscored_betas.npy'

#             if not exists(export_path):
#                 os.makedirs(export_path)         
            
#             # save the z scored map
#             np.save(export_filename, z_scored_array)
            
#             z_scored_array_mean = np.nanmean(z_scored_array)
#             z_scored_array_mean_list.append(z_scored_array_mean)
            
#             #print('z score mean',"{:.10f}".format(z_scored_array_mean))
            
#             mean_beta_raw = raw_beta_means[itemid][roi]
#             #print('raw mean',mean_beta_raw)
            
            
#             df_betas_perItem_currentRow = pd.DataFrame({'PartID': sub_uid,
#                                                  'localizer': localizer_id,
#                                                  'ROI': roi,
#                                                  'task': task_name,
#                                                  'session': session_num,
#                                                  'item_id': itemid,
#                                                  'model': glmsingle_model,
#                                                  'mean_beta_zscored': z_scored_array_mean,
#                                                  'mean_beta_raw': mean_beta_raw}, index=[0])
    
#             if not os.path.isfile(df_betas_perItem):
#                 df_betas_perItem_currentRow.to_csv(df_betas_perItem, index=False, header='column_names')
#             else: 
#                 df_betas_perItem_currentRow.to_csv(df_betas_perItem, mode='a', index=False, header=False)

